# Volatility Targeting — BTC / ETH

Stratégie de ciblage de volatilité appliquée aux prédictions LSTM.

- **Signal** : CMM — long si MA\_fast > MA\_slow, short sinon
- **Sizing** : `Position = Signal × vol_cible / vol_prédite_lissée`
- Crypto 24/7 — aucun filtre session
- Phases : paramètres défaut → grid search validation → meilleurs params test → synthèse

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from Backtester.Backtester import compute_backtest, plot_backtest
from prediction.settings import PAIRS, N_TARGETS, DATA_DIR, OUTPUT_DIR, CONTEXT_LENGTH

print(f'Imports OK  |  Paires : {PAIRS}')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')

In [ ]:
# ── Coûts de transaction (futures crypto, taker) ──────────────────────
TC_BPS = 5.0   # bps par unité de turnover

# ── Grilles hyperparamètres ───────────────────────────────────────────
FAST_MA_GRID   = [4, 8, 16, 32]        # en barres 15-min
SLOW_MA_GRID   = [48, 96, 192, 384]    # en barres 15-min
EWMA_SPAN_GRID = [4, 8, 16]

# ── Paramètres par défaut ─────────────────────────────────────────────
FAST_DEFAULT = 16
SLOW_DEFAULT = 96
EWMA_DEFAULT = 4

print('Configuration OK')

In [ ]:
def build_positions(preds_df, prices_df, fast, slow, ewma_span, vol_cible):
    '''Signal CMM × (vol_cible / vol_lissée) — sans filtre horaire (crypto 24/7).'''
    mf  = prices_df.rolling(fast, min_periods=1).mean()
    ms  = prices_df.rolling(slow, min_periods=1).mean()
    sig = pd.DataFrame(
        np.where(mf.values > ms.values, 1., -1.),
        index=prices_df.index, columns=prices_df.columns
    )
    vol_lissee = preds_df.abs().ewm(span=ewma_span, adjust=False).mean().clip(lower=1e-6)
    cidx = sig.index.intersection(vol_lissee.index)
    pos  = sig.loc[cidx] * (vol_cible / vol_lissee.loc[cidx])
    return pos.clip(-10, 10).reindex(prices_df.index, fill_value=0)


def get_metric(bt_obj, key, row=None):
    if bt_obj is None: return np.nan
    df   = bt_obj.metrics.toDataFrame()
    cols = [c for c in df.columns if key.lower() in c.lower()]
    if not cols: return np.nan
    r = row if (row and row in df.index) else df.index[0]
    try:
        return float(str(df.loc[r, cols[0]]).replace('%','').replace('bps','').strip())
    except Exception:
        return np.nan


def sharpe_fast(pos_df, perf_df, tc_bps=TC_BPS):
    common = pos_df.index.intersection(perf_df.index)
    bt = compute_backtest(
        pos_df.loc[common], perf_df.loc[common],
        transaction_costs_bps=tc_bps,
        compute_max_DD=False, show_pnl=False,
        position_lag_value=1, check_index=True, position_fill_value=0.0
    )
    row = 'Total' if (N_TARGETS > 1 and pos_df.shape[1] > 1) else None
    return get_metric(bt, 'sharpe', row=row)


def plot_pnl(pos_df, ret_df, title, val_s=None, test_s=None):
    common = pos_df.index.intersection(ret_df.index)
    pnl = (pos_df.loc[common] * ret_df.loc[common]).sum(axis=1).cumsum()
    fig, ax = plt.subplots(figsize=(14, 4))
    pnl.plot(ax=ax, color='steelblue', lw=0.8)
    if val_s  is not None: ax.axvline(val_s,  color='green',  ls='--', lw=1.2, label='Validation')
    if test_s is not None: ax.axvline(test_s, color='orange', ls='--', lw=1.2, label='Test')
    ax.axhline(0, color='black', lw=0.5, ls=':')
    ax.set_title(title); ax.set_ylabel('PnL cumulé (log-return)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


print('Fonctions utilitaires OK')

In [ ]:
# ── Chargement des prédictions ────────────────────────────────────────
preds = {}
for split in ('train', 'validation', 'test'):
    path = OUTPUT_DIR / f'{split}_lstm_1l_predictions.parquet'
    if not path.exists():
        raise FileNotFoundError(f'Prédictions manquantes : {path}  (lancer inference.ipynb ou quickrun.py)')
    df = pd.read_parquet(path)
    df.columns = PAIRS   # BTCUSDT_pred_bps → BTCUSDT
    preds[split] = df

train_preds = preds['train']
val_preds   = preds['validation']
test_preds  = preds['test']
all_preds   = pd.concat(list(preds.values())).sort_index()

train_start, train_end = train_preds.index[0],  train_preds.index[-1]
val_start,   val_end   = val_preds.index[0],    val_preds.index[-1]
test_start,  test_end  = test_preds.index[0],   test_preds.index[-1]

print(f'Train : {train_start.date()} → {train_end.date()}  ({len(train_preds):,} barres)')
print(f'Valid : {val_start.date()}   → {val_end.date()}  ({len(val_preds):,} barres)')
print(f'Test  : {test_start.date()}  → {test_end.date()}  ({len(test_preds):,} barres)')
print(f'\nPrédictions (BPS) :')
print(all_preds.describe().round(2))

In [ ]:
# ── Prix et rendements 15-min depuis les données brutes ───────────────
prices_list, returns_list = [], []
for crypto in PAIRS:
    df1m  = pd.read_parquet(DATA_DIR / f'{crypto}_1m.parquet')
    close = df1m.loc[df1m.index.minute % 15 == 0, 'close'].sort_index()
    ret   = np.log(close / close.shift(1))
    prices_list.append(close.rename(crypto))
    returns_list.append(ret.rename(crypto))

prices_df  = pd.concat(prices_list,  axis=1).sort_index()
returns_df = pd.concat(returns_list, axis=1).sort_index()

print(f'Prix    15m : {prices_df.shape}   {prices_df.index[0].date()} → {prices_df.index[-1].date()}')
print(f'Returns 15m : {returns_df.shape}')
print(f'\nStatistiques rendements (bps) :')
print((returns_df * 1e4).describe().round(2))

In [ ]:
# ── Phase 1 : Backtest avec paramètres par défaut ─────────────────────
vol_cible_default = all_preds.loc[train_start:train_end].abs().median()
print(f'Vol cible (médiane train, BPS) : {vol_cible_default.round(2).to_dict()}')

pos_default = build_positions(
    all_preds, prices_df,
    FAST_DEFAULT, SLOW_DEFAULT, EWMA_DEFAULT, vol_cible_default
)
print(f'Positions : {pos_default.shape}\n{pos_default.describe().round(3)}')

# Backtest sur le jeu de test
bt_default_test = compute_backtest(
    pos_default.loc[test_start:test_end],
    returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1,
    check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print('\n── Métriques — Défaut (test) ──────────────────────────────────')
print(bt_default_test.metrics.toDataFrame().to_string())
plot_pnl(pos_default, returns_df,
         f'Défaut (fast={FAST_DEFAULT}, slow={SLOW_DEFAULT}) — Historique',
         val_s=val_start, test_s=test_start)

In [ ]:
# ── Phase 2 : Grid Search sur la validation ───────────────────────────
val_preds_g   = all_preds.loc[val_start:val_end]
val_prices_g  = prices_df
val_returns_g = returns_df.loc[val_start:val_end]

vol_med = all_preds.loc[train_start:train_end].abs().median()
vol_q75 = all_preds.loc[train_start:train_end].abs().quantile(0.75)
vc_opts = {'median': vol_med, 'q75': vol_q75}

best_sharpe, best_params = -999., {}
results = []

n_total = sum(1 for f in FAST_MA_GRID for s in SLOW_MA_GRID if f < s) * len(EWMA_SPAN_GRID) * 2
n_done  = 0
for fast in FAST_MA_GRID:
    for slow in SLOW_MA_GRID:
        if fast >= slow: continue
        for ewma in EWMA_SPAN_GRID:
            for vc_key, vc in vc_opts.items():
                vpos = build_positions(val_preds_g, val_prices_g, fast, slow, ewma, vc)
                sh   = sharpe_fast(vpos.loc[val_start:val_end], val_returns_g)
                results.append({'fast': fast, 'slow': slow, 'ewma': ewma,
                                'vol': vc_key, 'sharpe': round(sh, 4)})
                if sh > best_sharpe:
                    best_sharpe = sh
                    best_params = {'fast': fast, 'slow': slow, 'ewma': ewma, 'vol': vc_key}
                n_done += 1
print(f'Grid search terminée ({n_done} combinaisons)')

df_res = pd.DataFrame(results).sort_values('sharpe', ascending=False)
print('\nTOP 10 (Sharpe net frais, validation) :')
print(df_res.head(10).to_string(index=False))
print(f'\nMeilleurs paramètres : {best_params}  (Sharpe val = {best_sharpe:.3f})')

In [ ]:
# ── Phase 3 : Backtest avec best_params ───────────────────────────────
vc_best  = vol_q75 if best_params['vol'] == 'q75' else vol_med
pos_best = build_positions(
    all_preds, prices_df,
    best_params['fast'], best_params['slow'], best_params['ewma'], vc_best
)

# Historique complet (train + val + test)
common = pos_best.index.intersection(returns_df.index)
bt_best_full = compute_backtest(
    pos_best.loc[common], returns_df.loc[common],
    detail_assets=True, position_lag_value=1,
    check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print('── Historique complet ──────────────────────────────────────────')
print(bt_best_full.metrics.toDataFrame().to_string())
plot_pnl(pos_best.loc[common], returns_df.loc[common],
         f'Best {best_params} — Historique complet',
         val_s=val_start, test_s=test_start)

# Test uniquement
bt_best_test = compute_backtest(
    pos_best.loc[test_start:test_end], returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1,
    check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print('\n── Test uniquement ────────────────────────────────────────────')
print(bt_best_test.metrics.toDataFrame().to_string())
plot_backtest(bt_best_test)

In [ ]:
# ── Phase 4 : Tableau de Synthèse ─────────────────────────────────────
METRICS = [
    ('Sharpe',    'sharpe'),
    ('Return/an', 'return (yearly)'),
    ('PnL/Trade', 'PnL/Trade'),
    ('Max DD',    'max DD'),
]

rows = []
for cfg, bt_obj in [('Défaut', bt_default_test), ('Optimisé', bt_best_test)]:
    for asset in PAIRS + ['Total']:
        r = {'Configuration': cfg, 'Actif': asset}
        for lbl, key in METRICS:
            r[lbl] = get_metric(bt_obj, key, row=asset)
        rows.append(r)

summary = pd.DataFrame(rows).set_index(['Configuration', 'Actif'])
print('=' * 65)
print('  SYNTHÈSE — Performances sur le jeu de test (net frais)')
print('=' * 65)
print(summary.to_string())